# 07 — Data Processing: IDF File Parsing

The `Tools` layer provides utilities for parsing Ivium's proprietary **IDF (Ivium Data Format)** files. It works entirely **offline** — no IviumSoft, no hardware, no driver required.

### What you can do

- Parse IDF files into Python lists/dicts
- Export data to CSV
- Batch-convert an entire directory of IDF files
- Visualize data with pandas and matplotlib

### IDF file structure

An IDF file can contain multiple data sections:

| Section key | Content |
|-------------|--------|
| `primary_data` | Main measurement (always present) — e.g. E/I pairs for LSV/CV, time/I/E for transients, Z1/Z2/freq for EIS |
| `ocpdata` | OCP measurement recorded before the experiment |
| `pretreatmentdata` | Pretreatment step data |
| `RsCs_data` | Rs/Cs measurement data |
| `osc_data` | Oscilloscope data (list of sections, one per sub-sweep) |

### Test data

This notebook uses IDF files from `tests/data/` — they're committed to the repository so you can run this notebook right now.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from pyvium import Tools

# Locate test data relative to this notebook
REPO_ROOT = Path("..")
TEST_DATA  = REPO_ROOT / "tests" / "data"

EIS_FILE  = TEST_DATA / "eis_test.idf"
OPEN_FILE = TEST_DATA / "test_open.idf"

print("Available test files:")
for f in TEST_DATA.iterdir():
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

## 1. `get_idf_data()` — Primary Data Only

The simplest function: extracts just the `primary_data` section and returns it as a flat list of rows.

In [ ]:
primary = Tools.get_idf_data(str(EIS_FILE))

print(f"Type    : {type(primary)}")
print(f"Points  : {len(primary)}")
print(f"Row[0]  : {primary[0]}")
print(f"Row[-1] : {primary[-1]}")

# Each row is a list of floats — column meaning depends on the method type
# For EIS: [Re(Z), Im(Z), frequency]
# For LSV: [potential, current, 0]
# For transients: [time, current, potential]

## 2. `get_all_idf_data()` — All Sections

Returns a dictionary with all sections present in the file. Keys that don't exist in the file are simply absent from the dict.

In [ ]:
all_data = Tools.get_all_idf_data(str(EIS_FILE))

print(f"Sections found: {list(all_data.keys())}")
for section, rows in all_data.items():
    if rows:
        print(f"  {section}: {len(rows)} point(s)")

## 3. EIS Data — Nyquist Plot

EIS `primary_data` rows are `[Re(Z), Im(Z), frequency]`.

In [ ]:
eis_df = pd.DataFrame(
    Tools.get_idf_data(str(EIS_FILE)),
    columns=["Z_re", "Z_im", "Frequency"]
)
print(eis_df.head())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Nyquist plot
axes[0].plot(eis_df["Z_re"], -eis_df["Z_im"], 'o-', markersize=4)
axes[0].set_xlabel("Z' (Ω)")
axes[0].set_ylabel("-Z'' (Ω)")
axes[0].set_title("Nyquist Plot")
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Bode plot — |Z| vs frequency
import numpy as np
Z_mag = np.sqrt(eis_df["Z_re"]**2 + eis_df["Z_im"]**2)
phase = np.degrees(np.arctan2(-eis_df["Z_im"], eis_df["Z_re"]))

ax_mag = axes[1]
ax_mag.semilogx(eis_df["Frequency"], Z_mag, 'b-o', markersize=3, label='|Z|')
ax_mag.set_xlabel("Frequency (Hz)")
ax_mag.set_ylabel("|Z| (Ω)", color='b')
ax_mag.set_title("Bode Plot")
ax_mag.grid(True, alpha=0.3)

ax_phase = ax_mag.twinx()
ax_phase.semilogx(eis_df["Frequency"], phase, 'r--o', markersize=3, label='Phase')
ax_phase.set_ylabel("Phase (°)", color='r')

plt.tight_layout()
plt.show()

## 4. OCP Data

If the IDF file includes an OCP pre-measurement, it appears in the `ocpdata` key. Rows are typically `[time, potential, current]`.

In [ ]:
all_data = Tools.get_all_idf_data(str(EIS_FILE))

if "ocpdata" in all_data and all_data["ocpdata"]:
    ocp_df = pd.DataFrame(all_data["ocpdata"])
    print(f"OCP data: {len(ocp_df)} points")
    print(ocp_df.head())

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(ocp_df.iloc[:, 0], ocp_df.iloc[:, 1], 'g-')
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("OCV (V)")
    ax.set_title("Open Circuit Potential (pre-measurement)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No OCP data in this file")

## 5. Inspecting All Sections

In [ ]:
for section, rows in all_data.items():
    if not rows:
        print(f"  {section}: empty")
        continue

    # osc_data is a list of sub-sections
    if section == "osc_data":
        print(f"  {section}: {len(rows)} sub-section(s)")
        for i, sub in enumerate(rows):
            print(f"    sub-section {i}: {len(sub)} points, {len(sub[0]) if sub else 0} columns")
    else:
        cols = len(rows[0]) if rows else 0
        print(f"  {section}: {len(rows)} points, {cols} columns")

## 6. Exporting to CSV

`export_to_csv(data, path)` writes any list of rows to a CSV file.

In [ ]:
import tempfile

# Export primary data to CSV
csv_path = os.path.join(tempfile.gettempdir(), "eis_primary.csv")
Tools.export_to_csv(primary, csv_path)
print(f"Exported to: {csv_path}")

# Verify by reading back
readback = pd.read_csv(csv_path, header=None)
print(f"Read back: {len(readback)} rows × {len(readback.columns)} columns")
print(readback.head())

## 7. `convert_idf_to_csv()` — One-liner Conversion

Reads an IDF file, extracts primary data, and writes a `.idf.csv` file next to the source file — all in one call.

In [ ]:
import shutil

# Work on a copy so we don't modify test data
tmp_dir  = Path(tempfile.mkdtemp())
tmp_idf  = tmp_dir / "eis_test.idf"
shutil.copy(EIS_FILE, tmp_idf)

Tools.convert_idf_to_csv(str(tmp_idf))

csv_out = tmp_idf.with_suffix(".idf.csv")
print(f"Created: {csv_out.name}  ({csv_out.stat().st_size} bytes)")

# Preview
df_check = pd.read_csv(csv_out, header=None)
print(df_check.head())

## 8. `convert_idf_dir_to_csv()` — Batch Conversion

Converts every `.idf` file in a directory. Returns a summary dict with counts and any filenames that failed.

In [ ]:
# Copy all test IDF files to a temp directory
batch_dir = Path(tempfile.mkdtemp())
for idf in TEST_DATA.glob("*.idf"):
    shutil.copy(idf, batch_dir / idf.name)

print(f"Converting {len(list(batch_dir.glob('*.idf')))} IDF files in {batch_dir}")

result = Tools.convert_idf_dir_to_csv(str(batch_dir))

print(f"\nResults:")
print(f"  Converted : {result['converted_count']}")
print(f"  Errors    : {result['error_count']}")
if result['errors']:
    print(f"  Failed    : {result['errors']}")

print("\nOutput files:")
for f in sorted(batch_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size} bytes)")

## 9. Building an Analysis Pipeline

Example: process a directory of EIS files and extract a single quantity (|Z| at 1 Hz) from each.

In [ ]:
import numpy as np
from pathlib import Path

def extract_z_at_freq(idf_path: str, target_freq: float) -> float:
    """Return |Z| at the frequency closest to target_freq."""
    data = Tools.get_idf_data(idf_path)
    df = pd.DataFrame(data, columns=["Z_re", "Z_im", "Frequency"])
    idx = (df["Frequency"] - target_freq).abs().idxmin()
    row = df.iloc[idx]
    return np.sqrt(row["Z_re"]**2 + row["Z_im"]**2)

# Run on all IDF files in the test directory
results = []
for idf_file in sorted(TEST_DATA.glob("*.idf")):
    try:
        z = extract_z_at_freq(str(idf_file), target_freq=1.0)
        results.append({"file": idf_file.name, "|Z| at 1 Hz (Ω)": z})
    except Exception as e:
        results.append({"file": idf_file.name, "|Z| at 1 Hz (Ω)": None})
        print(f"  Skipped {idf_file.name}: {e}")

summary = pd.DataFrame(results)
print(summary.to_string(index=False))

---

## Summary

| Task | Method | Notes |
|------|--------|-------|
| Read primary data | `Tools.get_idf_data(path)` | Returns `List[List[float]]` |
| Read all sections | `Tools.get_all_idf_data(path)` | Returns `Dict[str, List]` |
| Write CSV | `Tools.export_to_csv(data, path)` | Any list of rows |
| Convert one file | `Tools.convert_idf_to_csv(path)` | Creates `.idf.csv` alongside source |
| Batch convert dir | `Tools.convert_idf_dir_to_csv(dir)` | Returns summary dict |

### Data section column meanings

| Section | Col 0 | Col 1 | Col 2 |
|---------|-------|-------|-------|
| `primary_data` (EIS) | Re(Z) Ω | Im(Z) Ω | Frequency Hz |
| `primary_data` (LSV/CV) | Potential V | Current A | 0 |
| `primary_data` (transient) | Time s | Current A | Potential V |
| `ocpdata` | Time s | Potential V | Current A |

## Next

- **`08_batch_and_synchronization.ipynb`** — Coordinate multiple devices with the status parameter